In [54]:
import pandas as pd
print(pd.__file__)

C:\Users\Pooja\New folder\Lib\site-packages\pandas\__init__.py


In [55]:
import os
print(os.getcwd())

C:\Users\Pooja\Downloads


In [ ]:
2. Load data

In [56]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [57]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


In [ ]:
## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

In [58]:
price_df = pd.read_csv("./powerco_case/price_data (1).csv")
price_df["price_date"]= pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [59]:
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff ['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff= diff[['id', 'offpeak_diff_dec_january_energy', 'offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [60]:
df = pd.merge(df, diff, on='id')
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0,-0.004670,0.177779


In [64]:
df.nunique().to_frame()

,0
id,14606
channel_sales,8
cons_12m,11065
cons_gas_12m,2112
cons_last_month,4751
date_activ,1796
date_end,368
date_modif_prod,2129
date_renewal,386
forecast_cons_12m,13993


In [68]:
 (df['margin_gross_pow_ele'] == df['margin_net_pow_ele']).sum()

np.int64(14604)

In [72]:
df.shape

(14606, 45)

In [73]:
'margin_net_pow_ele' in df.columns

False

In [77]:
df['months_activ'] = ((df['date_end'] - df['date_activ']).dt.days / 30).round()
df[['date_activ', 'date_end', 'months_activ']].head()

,date_activ,date_end,months_activ
0,2013-06-15,2016-06-15,37.0
1,2009-08-21,2016-08-30,86.0
2,2010-04-16,2016-04-16,73.0
3,2010-03-30,2016-03-30,73.0
4,2010-01-13,2016-03-07,75.0


In [78]:
df['months_to_end'] = ((df['date_end'] - df['date_renewal']).dt.days / 30).round()
df['months_modif_prod'] = ((df['date_end'] - df['date_modif_prod']).dt.days / 30).round()
df['months_renewal'] = ((df['date_end'] - df['date_renewal']).dt.days / 30).round()
df[['months_to_end', 'months_modif_prod', 'months_renewal']].head()

,months_to_end,months_modif_prod,months_renewal
0,12.0,8.0,12.0
1,12.0,86.0,12.0
2,12.0,73.0,12.0
3,12.0,73.0,12.0
4,12.0,75.0,12.0


In [79]:
df = df.drop(columns=['months_to_end'])
df.shape

(14606, 48)

In [80]:
df['activ_year'] = df['date_activ'].dt.year
df['activ_month'] = df['date_activ'].dt.month
df[['date_activ', 'activ_year', 'activ_month']].head()

,date_activ,activ_year,activ_month
0,2013-06-15,2013,6
1,2009-08-21,2009,8
2,2010-04-16,2010,4
3,2010-03-30,2010,3
4,2010-01-13,2010,1


In [81]:
df['has_modif_prod'] = (df['date_modif_prod'] != df['date_activ']).astype(int)
df['has_modif_prod'].value_counts()

has_modif_prod
1    7334
0    7272
Name: count, dtype: int64

In [82]:
df['cons_vs_forecast'] = df['cons_12m'] - df['forecast_cons_12m']
df[['cons_12m', 'forecast_cons_12m', 'cons_vs_forecast']].head()

,cons_12m,forecast_cons_12m,cons_vs_forecast
0,0,0.00,0.00
1,4660,189.95,4470.05
2,544,47.96,496.04
3,1584,240.04,1343.96
4,4425,445.75,3979.25


In [ ]:
## Average price changes across periods

In [83]:
df['avg_cons_per_month'] = df['cons_12m'] / df['months_activ'].replace(0, 1)
df[['cons_12m', 'months_activ', 'avg_cons_per_month']].head()

,cons_12m,months_activ,avg_cons_per_month
0,0,37.0,0.000000
1,4660,86.0,54.186047
2,544,73.0,7.452055
3,1584,73.0,21.698630
4,4425,75.0,59.000000


In [84]:
df.shape
df.columns.tolist()

['id',
 'channel_sales',
 'cons_12m',
 'cons_gas_12m',
 'cons_last_month',
 'date_activ',
 'date_end',
 'date_modif_prod',
 'date_renewal',
 'forecast_cons_12m',
 'forecast_cons_year',
 'forecast_discount_energy',
 'forecast_meter_rent_12m',
 'forecast_price_energy_off_peak',
 'forecast_price_energy_peak',
 'forecast_price_pow_off_peak',
 'has_gas',
 'imp_cons',
 'margin_gross_pow_ele',
 'nb_prod_act',
 'net_margin',
 'num_years_antig',
 'origin_up',
 'pow_max',
 'var_year_price_off_peak_var',
 'var_year_price_peak_var',
 'var_year_price_mid_peak_var',
 'var_year_price_off_peak_fix',
 'var_year_price_peak_fix',
 'var_year_price_mid_peak_fix',
 'var_year_price_off_peak',
 'var_year_price_peak',
 'var_year_price_mid_peak',
 'var_6m_price_off_peak_var',
 'var_6m_price_peak_var',
 'var_6m_price_mid_peak_var',
 'var_6m_price_off_peak_fix',
 'var_6m_price_peak_fix',
 'var_6m_price_mid_peak_fix',
 'var_6m_price_off_peak',
 'var_6m_price_peak',
 'var_6m_price_mid_peak',
 'churn',
 'offpeak_dif

In [86]:
df.to_csv('clean_data_after_feature_engineering.csv', index=False)